In [ ]:
import datetime

# --- Data Configuration ---
MENU = {
    1: {"name": "Espresso", "price": 55},
    2: {"name": "Americano", "price": 60},
    3: {"name": "Latte", "price": 70},
    4: {"name": "Cappuccino", "price": 70},
    5: {"name": "Mocha", "price": 75},
    6: {"name": "Matcha Latte", "price": 80}
}

SIZE_OPTIONS = {
    "S": {"label": "Small", "extra": 0},
    "M": {"label": "Medium", "extra": 10},
    "L": {"label": "Large", "extra": 20}
}

MILK_OPTIONS = {
    1: {"label": "Regular Milk", "extra": 0},
    2: {"label": "Soy Milk", "extra": 10},
    3: {"label": "Almond Milk", "extra": 15}
}

SWEETNESS_LEVELS = ["0%", "25%", "50%", "100%"]

MEMBERS = {
    "0812345678": {"name": "Somchai", "discount_rate": 0.10},
    "0898765432": {"name": "Somsri", "discount_rate": 0.10}
}

# Daily sales record storage
sales_history = []


def display_menu():
    print("\n" + "="*45)
    print("         ☕ COFFEE SHOP MENU ☕")
    print("="*45)
    for key, item in MENU.items():
        print(f"  [{key}] {item['name']:<20} ฿{item['price']:.2f}")
    print("="*45)


def get_customization():
    # Select Size
    print("\n--- Select Size ---")
    for code, info in SIZE_OPTIONS.items():
        print(f"  [{code}] {info['label']} (+฿{info['extra']})")
    while True:
        size = input("Choose size (S/M/L): ").strip().upper()
        if size in SIZE_OPTIONS:
            break
        print(" Invalid size. Please choose S, M, or L.")

    # Select Milk
    print("\n--- Select Milk ---")
    for key, info in MILK_OPTIONS.items():
        print(f"  [{key}] {info['label']} (+฿{info['extra']})")
    while True:
        try:
            milk_choice = int(input("Choose milk option (1-3): "))
            if milk_choice in MILK_OPTIONS:
                break
        except ValueError:
            pass
        print(" Invalid option. Please enter 1, 2, or 3.")

    # Select Sweetness
    print("\n--- Select Sweetness Level ---")
    for idx, lvl in enumerate(SWEETNESS_LEVELS, 1):
        print(f"  [{idx}] {lvl}")
    while True:
        try:
            sw_choice = int(input("Choose sweetness (1-4): "))
            if 1 <= sw_choice <= len(SWEETNESS_LEVELS):
                sweetness = SWEETNESS_LEVELS[sw_choice - 1]
                break
        except ValueError:
            pass
        print(" Invalid option. Please enter 1 to 4.")

    return size, milk_choice, sweetness


def take_order():
    cart = []
    while True:
        display_menu()
        try:
            choice = int(input("\nSelect item number (0 to finish ordering): "))
            if choice == 0:
                if not cart:
                    print("Cart is empty. Returning to main menu.")
                    return None
                break
            if choice not in MENU:
                print(" Invalid item selection.")
                continue
        except ValueError:
            print(" Please enter a valid number.")
            continue

        item_base = MENU[choice]
        size, milk_choice, sweetness = get_customization()

        try:
            qty = int(input("\nEnter quantity: "))
            if qty <= 0:
                print(" Quantity must be at least 1.")
                continue
        except ValueError:
            print(" Invalid quantity.")
            continue

        # Calculate unit price
        unit_price = item_base['price'] + SIZE_OPTIONS[size]['extra'] + MILK_OPTIONS[milk_choice]['extra']
        total_item_price = unit_price * qty

        cart_item = {
            "name": item_base['name'],
            "size": size,
            "milk": MILK_OPTIONS[milk_choice]['label'],
            "sweetness": sweetness,
            "qty": qty,
            "unit_price": unit_price,
            "total_price": total_item_price
        }
        cart.append(cart_item)
        print(f"\n Added {qty}x {cart_item['name']} ({size}) to cart!")

        more = input("\nAdd more items? (y/n): ").strip().lower()
        if more != 'y':
            break

    return cart


def checkout(cart):
    if not cart:
        return

    subtotal = sum(item['total_price'] for item in cart)
    print("\n" + "="*45)
    print("             🛒 ORDER SUMMARY")
    print("="*45)
    for idx, item in enumerate(cart, 1):
        spec = f"{item['size']}, {item['milk']}, Sweet {item['sweetness']}"
        print(f"{idx}. {item['name']} ({spec})")
        print(f"   {item['qty']} x ฿{item['unit_price']:.2f} = ฿{item['total_price']:.2f}")
    print("-" * 45)
    print(f"Subtotal: ฿{subtotal:.2f}")

    # Membership check
    discount = 0.0
    member_name = "Guest"
    is_member = input("\nIs customer a member? (y/n): ").strip().lower()
    if is_member == 'y':
        phone = input("Enter member phone number: ").strip()
        if phone in MEMBERS:
            member_name = MEMBERS[phone]['name']
            discount = subtotal * MEMBERS[phone]['discount_rate']
            print(f" Member found: Welcome {member_name}! Discount (10%): -฿{discount:.2f}")
        else:
            print(" Member phone number not found. Proceeding as Guest.")

    final_total = subtotal - discount
    print(f"Grand Total: ฿{final_total:.2f}")
    print("="*45)

    # Payment processing
    while True:
        try:
            paid = float(input("\nEnter cash received: ฿"))
            if paid >= final_total:
                change = paid - final_total
                break
            else:
                print(f" Insufficient amount. Still need ฿{final_total - paid:.2f}")
        except ValueError:
            print(" Invalid payment amount.")

    # Generate Receipt
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("\n" + "*"*45)
    print("           ☕ GROUP15 EXPRESS COFFEE ☕")
    print("             OFFICIAL RECEIPT")
    print("*"*45)
    print(f"Date/Time: {now}")
    print(f"Customer : {member_name}")
    print("-" * 45)
    for item in cart:
        print(f"{item['qty']}x {item['name']:<15} [{item['size']}] ฿{item['total_price']:>8.2f}")
        print(f"   ({item['milk']}, {item['sweetness']})")
    print("-" * 45)
    print(f"Subtotal    : ฿{subtotal:>8.2f}")
    print(f"Discount    : -฿{discount:>7.2f}")
    print(f"Grand Total : ฿{final_total:>8.2f}")
    print(f"Cash Received: ฿{paid:>7.2f}")
    print(f"Change      : ฿{change:>8.2f}")
    print("*"*45)
    print("       Thank you for your order! ")
    print("*"*45 + "\n")

    # Save transaction
    sales_history.append({
        "timestamp": now,
        "items": cart,
        "total_amount": final_total
    })


def show_daily_summary():
    print("\n" + "="*45)
    print("        DAILY SALES SUMMARY REPORT")
    print("="*45)
    if not sales_history:
        print("No transactions recorded today yet.")
        print("="*45)
        return

    total_sales = sum(tx['total_amount'] for tx in sales_history)
    total_orders = len(sales_history)
    total_items_sold = sum(sum(item['qty'] for item in tx['items']) for tx in sales_history)

    print(f"Total Transactions : {total_orders}")
    print(f"Total Drinks Sold  : {total_items_sold} cups")
    print(f"Total Revenue      : ฿{total_sales:.2f}")
    print("="*45)


def main():
    while True:
        print("\n" + "═"*45)
        print("    ☕ SMART POS COFFEE TERMINAL SYSTEM ☕")
        print("═"*45)
        print("  [1] New Order (เปิดการสั่งซื้อใหม่)")
        print("  [2] Daily Sales Summary (สรุปยอดขายประจำวัน)")
        print("  [3] Exit System (ออกจากระบบ)")
        print("═"*45)
        choice = input("Select option (1-3): ").strip()

        if choice == '1':
            cart = take_order()
            if cart:
                checkout(cart)
        elif choice == '2':
            show_daily_summary()
        elif choice == '3':
            print("\n Exiting POS System. Have a great day!")
            break
        else:
            print(" Invalid option. Please select 1, 2, or 3.")


# Execute the program
if __name__ == "__main__":
    main()


═════════════════════════════════════════════
    ☕ SMART POS COFFEE TERMINAL SYSTEM ☕
═════════════════════════════════════════════
  [1] New Order (เปิดการสั่งซื้อใหม่)
  [2] Daily Sales Summary (สรุปยอดขายประจำวัน)
  [3] Exit System (ออกจากระบบ)
═════════════════════════════════════════════

         ☕ COFFEE SHOP MENU ☕
  [1] Espresso             ฿55.00
  [2] Americano            ฿60.00
  [3] Latte                ฿70.00
  [4] Cappuccino           ฿70.00
  [5] Mocha                ฿75.00
  [6] Matcha Latte         ฿80.00

--- Select Size ---
  [S] Small (+฿0)
  [M] Medium (+฿10)
  [L] Large (+฿20)

--- Select Milk ---
  [1] Regular Milk (+฿0)
  [2] Soy Milk (+฿10)
  [3] Almond Milk (+฿15)

--- Select Sweetness Level ---
  [1] 0%
  [2] 25%
  [3] 50%
  [4] 100%

 Added 1x Americano (L) to cart!

         ☕ COFFEE SHOP MENU ☕
  [1] Espresso             ฿55.00
  [2] Americano            ฿60.00
  [3] Latte                ฿70.00
  [4] Cappuccino           ฿70.00
  [5] Mocha              